# 04 — Statik Biyomekanik Hesaplama Motoru (Katman 3, G3, G4 & G5)
Eklem koordinatlarini 2B aci, segment yonelimi, bilateral asimetri, CoM izdusumu ve kosullu dis momente donusturur.
Bu katman yalnizca matematiksel hesap yapar; guvenilirlik karari vermez.

## Mimari Sozlesme ve Arastirma Ilkeleri

Bu notebook, **PhysioLoad AI — Olcum Duyarli Secici Biyomekanik Analiz Mimarisi**'nin bir parcasidir:
1. **Once olc, sonra guvenilirligini sinayarak raporla:** Her biyomekanik cikti bagimsiz degerlendirilir.
2. **Uc durumlu karar sozlesmesi:** `ACCEPTED` (Kabul), `REJECTED` (Ret), `NOT_COMPUTABLE` (Hesaplanamaz).
3. **Ayri dogrulama hatti:** Referans eklem verileri asla model tahminine girdi yapilmaz; yalnizca hata analizi ve A0-A4 karsilastirmasinda kullanilir.
4. **Veri sizintisini onleme:** Kisi ve deneme bazli boluntuleme (ayni katilimci train/val/test arasinda karismaz).
5. **Esik secimi:** Sayisal esikler yalnizca validation bolumunde belirlenir; test kumesi karsilastirma icin saklanir.


In [1]:
import sys
from pathlib import Path

# Proje kok dizini ve ortam kurulumu
here = Path.cwd().resolve()
code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))

from physioload_env import setup_environment
env = setup_environment(root=code_root)
print("Ortam Basariyla Yuklendi:")
print(env.summary())


Ortam Basariyla Yuklendi:
context       : vscode local
user          : berk-can
processed ver : v1
device        : cpu
code root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
artifacts root: C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
data root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai


In [2]:
from physioload.geometry import (
    joint_angle_degrees, segment_orientation_degrees,
    bilateral_asymmetry_degrees, SegmentMass, approximate_com_2d,
    external_moment_2d, force_from_mass_2d
)

# 1. 2B Eklem Acilari (G3)
hip = (320, 270)
knee = (300, 340)
ankle = (300, 420)
knee_angle = joint_angle_degrees(hip, knee, ankle)
print(f"Diz Acisi (2B): {knee_angle:.2f} derece")

# 2. Segment Yonelimi (G3)
shank_orient = segment_orientation_degrees(knee, ankle)
print(f"Baldır Yonelimi: {shank_orient:.2f} derece")

# 3. Bilateral Kinematik Asimetri (G4)
left_knee = 85.0
right_knee = 92.5
asym = bilateral_asymmetry_degrees(left_knee, right_knee)
print(f"Diz Sag-Sol Asimetrisi: {asym:.2f} derece")

# 4. Antropometrik Yaklasik 2B CoM Izdusumu (Winter 2009 katsayilari)
segments = [
    SegmentMass(proximal=(320, 120), distal=(320, 270), mass_fraction=0.50, com_fraction_from_proximal=0.45), # Govde
    SegmentMass(proximal=(320, 270), distal=(300, 420), mass_fraction=0.50, com_fraction_from_proximal=0.43), # Bacaklar
]
com_2d = approximate_com_2d(segments)
print(f"Yaklasik CoM Izdusumu (px): x={com_2d[0]:.1f}, y={com_2d[1]:.1f}")


Diz Acisi (2B): 164.05 derece
Baldır Yonelimi: 90.00 derece
Diz Sag-Sol Asimetrisi: 7.50 derece
Yaklasik CoM Izdusumu (px): x=315.7, y=261.0


In [3]:
# 5. Kosullu Dis Moment Hesabi (G5)
from physioload.physical_validity import PhysicalValidityContext, assess_physical_validity

# Metrik olcek (m/px) dogrulanmis ve 10 kg harici yuk tanimlanmis durum
load_px = (320, 200) # El hizasinda yuk
joint_px = knee      # Diz eklemi
force = force_from_mass_2d(mass_kg=10.0) # 10 kg * 9.80665 m/s2
scale_m_per_px = 0.002 # 1 px = 2 mm

valid_moment = external_moment_2d(joint_px, load_px, force, metres_per_pixel=scale_m_per_px)
print(f"Gecerli Kosulda Diz Dis Momenti: {valid_moment:.2f} N*m")

# Fiziksel gecerlilik denetimi (Metrik olcek yoksa)
invalid_ctx = PhysicalValidityContext(metric_scale_valid=False)
chk = assess_physical_validity('external_moment', invalid_ctx)
print(f"Olceksiz Gecerlilik Denetimi: computable={chk.computable}, Nedenler={chk.reasons}")


Gecerli Kosulda Diz Dis Momenti: 3.92 N*m
Olceksiz Gecerlilik Denetimi: computable=False, Nedenler=('metric_scale_not_validated', 'load_magnitude_missing', 'load_application_point_missing', 'load_direction_missing')
